# Tracking Electron Shower Origins

This notebook demonstrates how to identify the process that created electron showers in your eshower dataframe.

## Available Truth Information

Your eshower dataframe already contains truth matching information through `make_pfpdf`. For each shower PFParticle, you have access to:

- `primshw.truth.p.pdg` - PDG code of the matched true particle
- `primshw.truth.p.parent` - G4ID of the parent particle
- `primshw.truth.p.start_process` - Process that created this particle (e.g., "pi0Decay", "muonDecay", "deltaRay", "primary", etc.)
- `primshw.truth.p.end_process` - Process that ended this particle
- `primshw.truth.p.G4ID` - Geant4 ID of this particle
- `primshw.truth.p.startE` - Starting energy
- `primshw.truth.p.interaction_id` - Which neutrino interaction this came from

## Strategy

1. **Direct process identification**: Use `start_process` to identify common electron sources
2. **Parent tracking**: Use `parent` G4ID to look up parent particle information
3. **Chain tracking**: Follow parent chain back to find ultimate origin (pi0, muon decay, etc.)

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add workspace root to path
workspace_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('')))))
sys.path.insert(0, workspace_root)

from pyanalib.split_df_helpers import load_dfs

# Load your eshower dataframe
df_path = "/path/to/your/eshower.df"  # Update this path
eshower_dfs = load_dfs(df_path, keys2load=["eshower"], n_max_concat=100)
df = eshower_dfs["eshower"]

print(f"Dataframe shape: {df.shape}")
print(f"\nAvailable truth columns:")
truth_cols = [col for col in df.columns if 'truth' in str(col)]
for col in truth_cols[:20]:  # Show first 20
    print(f"  {col}")

## Method 1: Direct Process Identification

The simplest approach is to check the `start_process` field directly. Common processes that create electrons:

- `"pi0Decay"` - Electron from pi0 → γγ decay (one photon converts to e+e-)
- `"muonDecay"` - Electron from muon decay
- `"deltaRay"` - Delta ray (knock-on electron)
- `"primary"` - Primary electron from neutrino interaction
- `"compton"` - Compton scattering
- `"eBrem"` - Bremsstrahlung
- `"eIoni"` - Ionization
- `"conv"` - Pair conversion (γ → e+e-)

**Note**: For pi0 decays, the electron might come from a photon conversion, so you may see `"conv"` as the process, but the parent will be a photon from pi0 decay.

In [ ]:
# Method 1: Check start_process directly
# First, ensure you have good truth matching
good_match = (
    (df["primshw", "shw", "truth", "bestmatch", "energy_purity", "", ""] > 0.5) &
    (df["primshw", "shw", "truth", "bestmatch", "energy_completeness", "", ""] > 0.5)
)
df_matched = df[good_match].copy()

# Get the start process
start_process = df_matched["primshw", "truth", "p", "start_process", "", ""]

print("\n=== Direct Process Identification ===")
print(f"\nTotal matched showers: {len(df_matched)}")
print("\nProcess distribution:")
process_counts = start_process.value_counts()
for process, count in process_counts.items():
    print(f"  {process}: {count} ({count/len(df_matched)*100:.1f}%)")

# Categorize by origin type
df_matched["origin_type"] = "other"
df_matched.loc[start_process == "pi0Decay", "origin_type"] = "pi0_decay"
df_matched.loc[start_process == "muonDecay", "origin_type"] = "muon_decay"
df_matched.loc[start_process == "deltaRay", "origin_type"] = "delta_ray"
df_matched.loc[start_process == "primary", "origin_type"] = "primary_electron"
df_matched.loc[start_process == "conv", "origin_type"] = "photon_conversion"

print("\n=== Origin Type Summary ===")
print(df_matched["origin_type"].value_counts())

## Method 2: Parent Particle Tracking

For more detailed tracking, especially for photon conversions from pi0 decays, you need to look up the parent particle. The `parent` field contains the G4ID of the parent particle.

**Important**: To look up parent particles, you need to load the `rec.true_particles` table from the CAF file. This requires accessing the original file, not just the dataframe.

### Option A: Load true_particles table

If you have access to the original CAF file, you can load the true_particles table:

### Option B: Use interaction_id and process information

A simpler approach that doesn't require loading the full true_particles table is to use the `interaction_id` and `start_process` together. For pi0 decays:

1. Electrons from pi0 → γγ will have `start_process == "conv"` (photon conversion)
2. They'll share the same `interaction_id` as the pi0
3. You can check if there are photons (PDG=22) in the same interaction

However, this still requires access to the true_particles table to check for pi0s in the same interaction.

In [ ]:
# Method 2: Identify photon conversions (likely from pi0)
# Electrons from photon conversion will have start_process == "conv"
is_photon_conv = (df_matched["primshw", "truth", "p", "start_process", "", ""] == "conv")

print(f"\n=== Photon Conversions ===")
print(f"Showers from photon conversion: {is_photon_conv.sum()}")
print(f"\nThese are likely from:")
print(f"  - pi0 → γγ decays (most common)")
print(f"  - Other photon sources (less common)")

# To definitively identify pi0 parents, you'd need to:
# 1. Get the parent G4ID from df_matched["primshw", "truth", "p", "parent", "", ""]
# 2. Look up that parent in true_particles table
# 3. Check if parent is a photon (PDG=22)
# 4. Look up the photon's parent to see if it's a pi0 (PDG=111)

## Method 3: Practical Classification Without Full Parent Chain

For most analysis purposes, you can classify electron showers based on `start_process` and make reasonable inferences:

In [ ]:
# Method 3: Practical classification
def classify_electron_origin(row):
    """
    Classify electron shower origin based on available truth information.
    
    Returns:
        origin_category: str - One of 'primary', 'pi0_decay', 'muon_decay', 
                              'delta_ray', 'photon_conv', 'other'
    """
    start_process = row["primshw", "truth", "p", "start_process", "", ""]
    pdg = row["primshw", "truth", "p", "pdg", "", ""]
    
    # Ensure it's actually an electron
    if abs(pdg) != 11:
        return "not_electron"
    
    # Direct process identification
    if start_process == "primary":
        return "primary_electron"
    elif start_process == "pi0Decay":
        return "pi0_decay"
    elif start_process == "muonDecay":
        return "muon_decay"
    elif start_process == "deltaRay":
        return "delta_ray"
    elif start_process == "conv":
        # Photon conversion - most likely from pi0, but could be other sources
        return "photon_conversion"  # Likely pi0, but not guaranteed
    else:
        return "other"

# Apply classification
df_matched["origin_category"] = df_matched.apply(classify_electron_origin, axis=1)

print("\n=== Electron Origin Classification ===")
print(df_matched["origin_category"].value_counts())
print("\n\nNote: 'photon_conversion' category likely includes pi0 decays,")
print("but to be certain, you'd need to track the parent chain.")

## Method 4: Full Parent Chain Tracking (Requires CAF File Access)

To definitively track back to pi0 decays or muon decays, you need to:

1. Load the `rec.true_particles` table
2. Use the `parent` G4ID to look up parent particles
3. Follow the chain back until you find a pi0 (PDG=111) or muon (PDG=±13)

Here's a function to do this:

In [ ]:
def track_to_ultimate_parent(electron_g4id, parent_g4id, trueparticles_df, max_depth=10):
    """
    Track parent chain back to find ultimate origin (pi0, muon, etc.).
    
    Parameters:
    -----------
    electron_g4id : int
        G4ID of the electron
    parent_g4id : int
        G4ID of the electron's parent
    trueparticles_df : pd.DataFrame
        DataFrame with true_particles table, indexed by (entry, G4ID)
    max_depth : int
        Maximum depth to search (prevent infinite loops)
    
    Returns:
    --------
    dict with keys:
        'ultimate_parent_pdg': PDG code of ultimate parent
        'ultimate_parent_process': Process that created ultimate parent
        'chain_length': Number of generations
        'is_pi0': True if from pi0 decay
        'is_muon': True if from muon decay
    """
    if pd.isna(parent_g4id) or parent_g4id < 0:
        return {
            'ultimate_parent_pdg': None,
            'ultimate_parent_process': None,
            'chain_length': 0,
            'is_pi0': False,
            'is_muon': False
        }
    
    current_g4id = parent_g4id
    depth = 0
    
    # Get entry from electron (assuming same entry)
    # You'll need to pass entry separately or extract from index
    
    while depth < max_depth:
        # Look up current parent
        # Note: This requires matching on (entry, G4ID)
        # parent_row = trueparticles_df.loc[(entry, current_g4id)]
        
        # Check if it's a pi0 or muon
        # parent_pdg = parent_row['pdg']
        # if abs(parent_pdg) == 111:  # pi0
        #     return {'ultimate_parent_pdg': 111, 'is_pi0': True, ...}
        # elif abs(parent_pdg) == 13:  # muon
        #     return {'ultimate_parent_pdg': 13, 'is_muon': True, ...}
        
        # Move to next parent
        # current_g4id = parent_row['parent']
        depth += 1
        
        # Break if no more parents
        # if pd.isna(current_g4id) or current_g4id < 0:
        #     break
    
    # Placeholder return
    return {
        'ultimate_parent_pdg': None,
        'ultimate_parent_process': None,
        'chain_length': depth,
        'is_pi0': False,
        'is_muon': False
    }

print("\nNote: Full implementation requires access to true_particles table.")
print("The structure depends on how you load and index the CAF file.")

## Recommendations

### For Quick Analysis:
1. Use **Method 1** (direct `start_process` check) - this is the simplest and works with just your dataframe
2. Use **Method 3** (practical classification) - combines process info for categorization

### For Detailed Tracking:
1. Use **Method 4** (parent chain tracking) - requires loading `rec.true_particles` from CAF file
2. You'll need to modify your config to also load the true_particles table, or access it separately

### Do You Need to Map PFP Back to Initial PFP?

**Short answer: Usually no.** The truth matching already gives you the true particle that best matches your reconstructed shower. The `parent` field in the truth information refers to the **true particle parent**, not the PFP parent.

However, if you want to understand the **reconstruction hierarchy** (which PFP is the parent in the slice), you would use:
- `pfp.parent` - PFP ID of the parent PFParticle in the reconstruction
- This tells you the reconstruction hierarchy, not the truth origin

For understanding **physical origin** (pi0 decay, muon decay, etc.), use the truth matching fields (`primshw.truth.p.*`), not the PFP hierarchy.

## Example: Complete Workflow

Here's a complete example combining the methods:

In [ ]:
# Complete workflow example

# 1. Load dataframe (already done above)
# df = ...

# 2. Apply your electron shower selection cuts
# df_selected = df[your_cuts].copy()

# 3. Require good truth matching
good_match = (
    (df["primshw", "shw", "truth", "bestmatch", "energy_purity", "", ""] > 0.5) &
    (df["primshw", "shw", "truth", "bestmatch", "energy_completeness", "", ""] > 0.5)
)
df_truth = df[good_match].copy()

# 4. Require it's actually an electron
is_electron = df_truth["primshw", "truth", "p", "pdg", "", ""].abs() == 11
df_electrons = df_truth[is_electron].copy()

# 5. Classify origins
df_electrons["origin_category"] = df_electrons.apply(classify_electron_origin, axis=1)

# 6. Summary statistics
print("\n=== Electron Shower Origin Summary ===")
print(f"Total electron showers: {len(df_electrons)}")
print("\nOrigin breakdown:")
for origin, count in df_electrons["origin_category"].value_counts().items():
    print(f"  {origin}: {count} ({count/len(df_electrons)*100:.1f}%)")

# 7. Analyze by origin type
print("\n=== Energy distributions by origin ===")
for origin in df_electrons["origin_category"].unique():
    origin_df = df_electrons[df_electrons["origin_category"] == origin]
    energy_col = ("primshw", "shw", "maxplane_energy", "", "", "")
    if energy_col in origin_df.columns:
        print(f"\n{origin}:")
        print(f"  Mean energy: {origin_df[energy_col].mean():.2f} GeV")
        print(f"  Median energy: {origin_df[energy_col].median():.2f} GeV")

## References

- Truth branches are defined in `makedf/branches.py` - see `trueparticlenames`
- PFP truth matching is done in `make_pfpdf()` in `makedf/makedf.py`
- Your eshower config loads this through `make_eshowerdf_mc()`

## Common Process Names

From Geant4 physics processes:
- `"primary"` - Primary particle from generator
- `"pi0Decay"` - Direct pi0 decay
- `"muonDecay"` - Muon decay
- `"deltaRay"` - Delta ray (knock-on electron)
- `"conv"` - Pair conversion (γ → e+e-)
- `"compton"` - Compton scattering
- `"eBrem"` - Bremsstrahlung
- `"eIoni"` - Ionization
- `"phot"` - Photoelectric effect
- `"annihil"` - Positron annihilation